# 01 — Exploratorní analýza dat (EDA)

Cílem tohoto notebooku je porozumět struktuře a charakteristikám datasetu před modelováním.  
Analyzujeme **rozdělení příznaků**, **korelační vztahy** a **vyváženost cílové třídy `Dropout`**.

**Dataset:** Student Dropout Dataset v3 — 10 000 studentů, 19 příznaků.  
**Cílová proměnná:** `Dropout` (1 = student studium opustil, 0 = student dostudoval)  
**Zdroj:** Kaggle / OpenML — syntetický dataset modelující faktory odchodu studentů z VŠ.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv('../data/student_dropout_dataset_v3.csv')

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


## 1. Rozdělení numerických příznaků

Histogramy ukazují, jak jsou hodnoty jednotlivých příznaků rozloženy napříč celým datasetem.  
KDE křivka (Kernel Density Estimate) znázorňuje hustotu pravděpodobnosti.


In [2]:

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('Student_ID', errors='ignore')

fig, axes = plt.subplots(int(np.ceil(len(numeric_cols) / 3)), 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i], color='#0077BB')
    axes[i].set_title(col)
    axes[i].set_ylabel('Počet')
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.suptitle('Distribuce numerických příznaků studentů',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

### Interpretace histogramů

- **CGPA, GPA, Semester_GPA** vykazují přibližně normální rozdělení s mírným levým šikmem — většina studentů dosahuje průměrných hodnot okolo 2,5–3,0.
- **Stress_Index** je rozložen rovnoměrně v rozsahu 1–10, bez výrazné dominance krajních hodnot.
- **Study_Hours_per_Day** má bimodální charakter — část studentů studuje méně než 2 hodiny denně, část přes 5 hodin.
- **Attendance_Rate** vykazuje silnou koncentraci v horním pásmu (80–100 %), s výrazným poklesem u nízkých hodnot.
- **Family_Income** a **Travel_Time_Minutes** mají pravostranné šikmé rozdělení — většina hodnot je nízká, ale existují výrazné odlehlé hodnoty (outlieři).
- **Assignment_Delay_Days**: levostranné rozdělení — většina studentů odevzdává úkoly bez nebo s minimálním zpožděním.

**Klíčový závěr:** Žádný příznak nevyžaduje logaritmickou transformaci. Příznaky jako `Family_Income` a `Stress_Index` budou normalizovány StandardScalerem v předzpracování.


## 2. Korelační matice

Korelační matice zachycuje lineární vztahy mezi numerickými příznaky.  
Hodnoty blízké **+1** označují silnou pozitivní korelaci, hodnoty blízké **−1** silnou negativní korelaci.


In [3]:
plt.figure(figsize=(12, 10))

corr_matrix = df[numeric_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            vmax=1, vmin=-1, center=0, square=True, linewidths=.5,
            cbar_kws={"shrink": .8})

plt.title('Korelační matice parametrů studentů', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks()
plt.tight_layout()
plt.show()

### Interpretace korelační matice

**Korelace s cílovou proměnnou `Dropout`:**

| Příznak | Korelace | Interpretace |
|---|---|---|
| `CGPA` | −0,445 | Studenti s nižším kumulativním průměrem mají výrazně vyšší riziko odchodu |
| `GPA` / `Semester_GPA` | −0,460 / −0,445 | Totožný signál — výsledky v aktuálním semestru jsou stejně silným prediktorem |
| `Stress_Index` | +0,256 | Vyšší stres koreluje s odchodem ze studia |
| `Attendance_Rate` | −0,164 | Nižší docházka signalizuje riziko dropoutu |
| `Study_Hours_per_Day` | −0,089 | Slabší signál — studijní čas sám o sobě nestačí |
| `Family_Income` | −0,011 | Téměř žádný lineární vztah |

**Multikolinearita:**  
`CGPA`, `GPA` a `Semester_GPA` jsou navzájem silně korelovány (r ≈ 0,96–1,00) — jde prakticky o totéž měření zachycené třemi různými způsoby. V předzpracování proto zachováme pouze `CGPA` a odvozený příznak `GPA_trend = CGPA − Semester_GPA`, který zachycuje *trend* ve výsledcích (pozitivní = student se zhoršuje, negativní = zlepšuje). `GPA` a `Semester_GPA` budou z datasetu odstraněny.

**Klíčový závěr:** Nejsilnějšími lineárními prediktory dropoutu jsou akademické výsledky (`CGPA`) a stres (`Stress_Index`). Stromové modely navíc dokáží zachytit nelineární interakce mezi příznaky.


## 3. Rozdělení cílové proměnné — nevyváženost tříd

Vizualizace četností tříd `Dropout` a identifikace případné nevyváženosti datasetu.


In [4]:
plt.figure(figsize=(8, 5))

sns.countplot(data=df, x='Dropout', hue='Dropout', palette='viridis', legend=False)

plt.title('Rozložení cílové třídy: Nevyvážený dataset', fontsize=14, pad=15)
plt.ylabel('Počet studentů')
plt.xlabel('Status studenta (0 = Dostuduje, 1 = Dropout)')
plt.xticks()
plt.yticks()

plt.tight_layout()
plt.show()

### Interpretace nevyváženosti tříd

Dataset obsahuje **10 000 studentů**:
- **Třída 0 (Dostuduje):** 7 646 studentů — **76,5 %**
- **Třída 1 (Dropout):** 2 354 studentů — **23,5 %**

Poměr tříd je přibližně **3,25 : 1** ve prospěch majoritní třídy. Jde o **středně nevyvážený dataset** — model trénovaný na surových datech by se naučil predikovat převážně majoritní třídu a dosahoval by opticky vysoké Accuracy (≈ 76,5 %) při téměř nulovém Recall pro třídu Dropout.

**Důsledky pro modelování:**
1. **Metrika:** Accuracy není vhodná — jako primární metriku použijeme **Recall** pro třídu Dropout a **ROC-AUC**.
2. **Předzpracování:** Pro vyrovnání tříd použijeme **SMOTE** (Synthetic Minority Oversampling Technique) aplikovaný výhradně na trénovací množinu v rámci pipeline (bez data leakage).
3. **Baseline:** Dummy klasifikátor predikující vždy majoritní třídu dosáhne Accuracy ≈ 76,5 % — tato hodnota slouží jako spodní hranice pro porovnání.
